# 06 — Custom Tool Creation

Build custom tools and bind them to a ReAct agent.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from datetime import datetime, timedelta
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

## Define Tools

In [ ]:
@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

class WeatherInput(BaseModel):
    city: str = Field(description="City name")
    units: str = Field(default="celsius", description="Temperature units")

@tool(args_schema=WeatherInput)
def get_weather(city: str, units: str = "celsius") -> str:
    """Get the current weather for a city."""
    weather_data = {
        "london": {"temp_c": 12, "condition": "Cloudy"},
        "new york": {"temp_c": 22, "condition": "Sunny"},
        "tokyo": {"temp_c": 28, "condition": "Humid"},
        "sydney": {"temp_c": 18, "condition": "Partly cloudy"},
        "paris": {"temp_c": 15, "condition": "Rainy"},
    }
    data = weather_data.get(city.lower(), {"temp_c": 20, "condition": "Unknown"})
    temp = data["temp_c"]
    if units == "fahrenheit":
        temp = round(temp * 9 / 5 + 32)
        unit_str = "°F"
    else:
        unit_str = "°C"
    return f"{city}: {temp}{unit_str}, {data['condition']}"

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "JPY": 149.50, "AUD": 1.53}
    from_curr = from_currency.upper()
    to_curr = to_currency.upper()
    if from_curr not in rates or to_curr not in rates:
        return f"Unsupported currency. Supported: {', '.join(rates.keys())}"
    usd_amount = amount / rates[from_curr]
    result = usd_amount * rates[to_curr]
    return f"{amount} {from_curr} = {result:.2f} {to_curr}"

@tool
def date_calculator(operation: str, days: int) -> str:
    """Calculate dates relative to today. operation: 'add' or 'subtract'."""
    today = datetime.now()
    result = today - timedelta(days=days) if operation == "subtract" else today + timedelta(days=days)
    return f"Result: {result.strftime('%A, %B %d, %Y')}"

## Run the Agent

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(llm, [get_current_time, get_weather, convert_currency, date_calculator])

queries = [
    "What time is it right now?",
    "What's the weather like in Tokyo and London?",
    "Convert 100 USD to EUR and JPY",
    "What date will it be 45 days from now?",
]

for q in queries:
    print(f"Q: {q}")
    response = agent.invoke({"messages": [HumanMessage(content=q)]})
    print(f"A: {response['messages'][-1].content}\n")